# Cells and boundary conditions

A `Cell` defines the orthorhombic simulation domain. Lengths and origins
are in meters; `tangle.units` supplies `um`, `mm`, and `nm` as plain
float multipliers. `periodic` names the axes that use periodic
minimum-image contact; every other axis is bounded by hard walls. It
accepts a string of axis letters such as `"xy"`, a single axis, or an
`[x, y, z]` bool triple.

In [ ]:
import tangle
from tangle.units import mm, nm, um

# Units are ordinary floats in meters, so they compose with arithmetic.
print(7 * um, 1.5 * mm, 250 * nm)

# Omitting `periodic` gives three bounded axes with the origin at zero.
bounded = tangle.Cell([1 * mm, 2 * mm, 3 * mm])
# This sheet repeats in x and y but has bounded through-thickness
# surfaces; the origin centers the x/y coordinates.
periodic_sheet = tangle.Cell(
    [1 * mm, 1 * mm, 2 * mm],
    periodic="xy",
    origin=[-0.5 * mm, -0.5 * mm, 0.0],
)
{
    "lengths": periodic_sheet.lengths,
    "periodic": periodic_sheet.periodic,
    "origin": periodic_sheet.origin,
    "stack_axis": periodic_sheet.stack_axis,
}

## The stack axis

Layer-aware operations (layered generation, layer placement, needling,
and default compaction) act along one *stack axis*. When exactly one
axis is bounded, the cell infers it as the stack axis, so a
`periodic="xy"` sheet stacks along z. Otherwise the stack axis
defaults to z; pass `stack_axis=` to the `Cell` (or to `Recipe`) to
choose another. Axes may be written as `"x"`, `"y"`, `"z"` or `0`,
`1`, `2`; the property always reports the index.

In [ ]:
# A bool triple is equivalent to the axis-letter string.
same_sheet = tangle.Cell([1 * mm, 1 * mm, 2 * mm], periodic=[True, True, False])
# y is the only bounded axis here, so it is inferred as the stack axis.
wall_in_y = tangle.Cell([1 * mm, 2 * mm, 1 * mm], periodic="xz")
# A fully periodic box has no bounded axis, so name the stack axis.
bulk = tangle.Cell([1 * mm, 1 * mm, 1 * mm], periodic="xyz", stack_axis="x")
{
    "same flags": same_sheet.periodic == periodic_sheet.periodic,
    "wall_in_y": wall_in_y.stack_axis,
    "bulk": bulk.stack_axis,
    "bounded box": bounded.stack_axis,
}

## Boundary interpretation

- Periodic axes wrap cell-list neighborhoods and use minimum-image
  capsule separation. They are useful for representative in-plane areas.
- Bounded axes enforce planar walls. Use these for exposed surfaces or
  through-thickness compaction.
- Cell coordinates describe placed geometry only; rest centerlines are
  intrinsic material data and need not lie inside the cell.